In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install xgboost -q

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

df_clean = pd.read_csv('/content/drive/MyDrive/software-build-failure-prediction/data/processed/df_clean.csv')
print(df_clean.shape)
df_clean.head()

(252151, 28)


,git_branch,git_prev_commit_resolution_status,gh_is_pr,gh_pull_req_num,gh_lang,gh_team_size,gh_num_issue_comments,gh_num_commit_comments,gh_num_pr_comments,git_diff_src_churn,...,gh_test_lines_per_kloc,gh_test_cases_per_kloc,gh_asserts_cases_per_kloc,gh_by_core_team_member,gh_description_complexity,gh_repo_age,gh_repo_num_commits,tr_build_number,gh_build_started_at,target
0,master,merge_found,False,0.0,ruby,31,0.0,0,0.0,117,...,1539.431157,163.865546,225.921138,False,21.0,655.82,1139,23,2011-04-16 11:24:39,0
1,master,merge_found,False,0.0,ruby,6,0.0,1,0.0,2,...,804.648241,50.251256,90.452261,True,21.0,1009.77,767,19,2011-04-16 17:37:39,0
2,master,build_found,False,0.0,ruby,32,0.0,0,0.0,0,...,1534.966162,163.390267,225.265872,True,21.0,656.13,1141,24,2011-04-16 19:25:03,1
3,master,build_found,False,0.0,ruby,32,0.0,0,0.0,0,...,1534.966162,163.390267,225.265872,True,21.0,656.15,1142,25,2011-04-16 19:40:07,0
4,master,build_found,False,0.0,ruby,32,0.0,0,0.0,0,...,1534.966162,163.390267,225.265872,True,21.0,656.16,1143,26,2011-04-16 19:54:28,0


In [3]:
df_clean = df_clean.sort_values('gh_build_started_at')

split_idx = int(len(df_clean) * 0.8)
train_df = df_clean.iloc[:split_idx]
test_df = df_clean.iloc[split_idx:]

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
print('Train failure rate:', train_df['target'].mean())
print('Test failure rate:', test_df['target'].mean())

Train shape: (201720, 28)
Test shape: (50431, 28)
Train failure rate: 0.22582788023002182
Test failure rate: 0.1734449049195931


In [4]:
X_train = train_df.drop(columns=['target', 'gh_build_started_at'])
y_train = train_df['target']
X_test = test_df.drop(columns=['target', 'gh_build_started_at'])
y_test = test_df['target']

print(X_train.shape, X_test.shape)

(201720, 26) (50431, 26)


In [5]:
categorical_cols = X_train.select_dtypes(include='object').columns.tolist()
print('Categorical columns:', categorical_cols)

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    X_test[col] = X_test[col].astype(str).map(lambda x: x if x in le.classes_ else le.classes_[0])
    X_test[col] = le.transform(X_test[col])
    encoders[col] = le

print('Encoding done.')

Categorical columns: ['git_branch', 'git_prev_commit_resolution_status', 'gh_lang']
Encoding done.


In [6]:
numeric_cols = X_train.select_dtypes(include='number').columns.tolist()

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

print('Scaling done.')

Scaling done.


In [7]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print('scale_pos_weight:', scale_pos_weight)

scale_pos_weight: 3.428151205163103


In [8]:
lr_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

rf_model = RandomForestClassifier(class_weight='balanced', n_estimators=200, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

xgb_model = XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric='logloss', random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)

print('All 3 models trained.')

All 3 models trained.


In [10]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:,1]
    results = {
        'Model': name,
        'Pipeline': 'Clean',
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_proba)
    }
    return results

results_list = []
results_list.append(evaluate_model('Logistic Regression', lr_model, X_test, y_test))
results_list.append(evaluate_model('Random Forest', rf_model, X_test, y_test))
results_list.append(evaluate_model('XGBoost', xgb_model, X_test, y_test))

clean_results_df = pd.DataFrame(results_list)
print(clean_results_df)

                 Model Pipeline  Precision    Recall        F1   AUC-ROC
0  Logistic Regression    Clean   0.228542  0.599977  0.331000  0.594406
1        Random Forest    Clean   0.787346  0.102435  0.181285  0.640256
2              XGBoost    Clean   0.270030  0.524408  0.356493  0.651884


In [11]:
existing_results = pd.read_csv('/content/drive/MyDrive/software-build-failure-prediction/results/metrics_comparison.csv')
combined_results = pd.concat([existing_results, clean_results_df], ignore_index=True)

combined_results.to_csv('/content/drive/MyDrive/software-build-failure-prediction/results/metrics_comparison.csv', index=False)
print(combined_results)

                 Model Pipeline  Precision    Recall        F1   AUC-ROC
0  Logistic Regression    Leaky   0.385350  0.784269  0.516780  0.769678
1        Random Forest    Leaky   0.819483  0.779010  0.798734  0.936568
2              XGBoost    Leaky   0.670291  0.852750  0.750591  0.947054
3  Logistic Regression    Clean   0.228542  0.599977  0.331000  0.594406
4        Random Forest    Clean   0.787346  0.102435  0.181285  0.640256
5              XGBoost    Clean   0.270030  0.524408  0.356493  0.651884


In [12]:
import joblib
joblib.dump(rf_model, '/content/drive/MyDrive/software-build-failure-prediction/results/rf_clean_model.pkl')
joblib.dump(xgb_model, '/content/drive/MyDrive/software-build-failure-prediction/results/xgb_clean_model.pkl')
joblib.dump(list(X_train.columns), '/content/drive/MyDrive/software-build-failure-prediction/results/clean_feature_names.pkl')
print('Models saved.')

Models saved.
